# Tutorial 02: PDF Processing and Table Extraction

This notebook demonstrates how to process PDFs and extract GDP tables using Tabula.

## What You'll Learn

1. Organize downloaded PDFs by year
2. Extract tables from PDFs using Tabula
3. Handle different PDF formats (scanned vs digital)
4. Generate input PDFs with only relevant pages
5. Inspect and validate extracted tables

## Setup and Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from peru_gdp_rtd.config import get_settings
from peru_gdp_rtd.processors import extract_table, organize_files_by_year
from peru_gdp_rtd.processors.pdf_processor import generate_input_pdfs

settings = get_settings("../config/config.yaml")
print(f"Project: {settings.project.name}")

## Step 1: Organize PDFs by Year

In [ ]:
# Organize downloaded PDFs into year-based folders
file_dict = organize_files_by_year(
    new_wr_folder=settings.paths.new_wr,
    old_wr_folder=settings.paths.old_wr,
)

print("Files organized by year:")
for year in sorted(file_dict.keys()):
    print(f"  {year}: {len(file_dict[year])} files")

## Step 2: Extract Table from a Sample PDF

In [ ]:
# Get a sample PDF (most recent year)
if file_dict:
    latest_year = max(file_dict.keys())
    sample_pdf = file_dict[latest_year][0]

    print(f"Sample PDF: {sample_pdf.name}")
    print(f"Year: {latest_year}")
    print(f"Path: {sample_pdf}")

In [ ]:
# Extract tables from specific pages
# GDP tables are typically on pages 3-4
tables = extract_table(
    pdf_path=sample_pdf,
    pages=[3, 4],
    area=None,  # Auto-detect table area
)

print(f"Extracted {len(tables)} tables")

# Display first table
if tables:
    print("\nFirst table preview:")
    display(tables[0].head(10))

## Step 3: Inspect Table Structure

In [ ]:
if tables:
    df = tables[0]

    print("Table Information:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Dtypes:\n{df.dtypes}")
    print(f"\nFirst few rows:")
    display(df.head())

## Step 4: Extract with Custom Area

In [ ]:
# For better precision, specify extraction area
# Area coordinates: [top, left, bottom, right] in points
custom_area = [[100, 50, 500, 750]]

tables_custom = extract_table(
    pdf_path=sample_pdf,
    pages=[3],
    area=custom_area,
)

if tables_custom:
    print("Custom area extraction:")
    display(tables_custom[0].head())

## Step 5: Compare OLD vs NEW PDFs

In [ ]:
# OLD PDFs (pre-2013): Scanned images, lower quality
# NEW PDFs (2013+): Digital tables, higher quality

old_files = [f for year, files in file_dict.items() if year < 2013 for f in files]
new_files = [f for year, files in file_dict.items() if year >= 2013 for f in files]

print(f"OLD PDFs (pre-2013): {len(old_files)} files")
print(f"NEW PDFs (2013+): {len(new_files)} files")

# Note: OLD PDFs often require OCR and more complex processing

## Step 6: Generate Input PDFs

In [ ]:
# Generate input PDFs (extract only relevant pages)
# This reduces file size and processing time

# Uncomment to run:
# generate_input_pdfs(
#     source_folder=settings.paths.new_wr,
#     output_folder=settings.paths.data_root / 'input',
#     pages=[3, 4],  # GDP tables are on pages 3-4
# )

print("To generate input PDFs, uncomment the code above.")
print("Or use: python scripts/update_rtd.py --steps 2")

## Step 7: Batch Process Multiple PDFs

In [ ]:
from tqdm import tqdm

# Process first 5 PDFs from latest year
if file_dict:
    latest_year = max(file_dict.keys())
    pdfs_to_process = file_dict[latest_year][:5]

    results = []

    for pdf_path in tqdm(pdfs_to_process, desc="Extracting tables"):
        try:
            tables = extract_table(pdf_path, pages=[3, 4])
            results.append(
                {"file": pdf_path.name, "tables_extracted": len(tables), "status": "success"}
            )
        except Exception as e:
            results.append(
                {"file": pdf_path.name, "tables_extracted": 0, "status": f"error: {str(e)}"}
            )

    # Display results
    results_df = pd.DataFrame(results)
    display(results_df)

## Key Takeaways

1. **Tabula-py**: Powerful tool for PDF table extraction
2. **Page Selection**: Extract only relevant pages to save time
3. **Area Specification**: Custom areas for better precision
4. **OLD vs NEW**: Different processing strategies for different PDF types
5. **Batch Processing**: Process multiple PDFs efficiently

## Next Steps

- **Tutorial 03**: Data Cleaning - Standardize extracted tables
- **Tutorial 04**: RTD Construction - Build vintage datasets

## Common Issues

- **Empty tables**: Check page numbers and extraction area
- **Malformed data**: Try different Tabula options (lattice vs stream)
- **Java errors**: Ensure JRE is installed